# RoboQuest 2026 — クイックスタート編
まずロボットを動かしてみましょう。スライダーで設定を変えられます。仕組みを学びたいときは [解説・実験編](guide_and_experiments_ja.ipynb) へ。

Google Colabで上から順に実行します。初回はライブラリとロボットモデルをダウンロードします。学習には時間がかかります。

**流れ：セットアップ → 保存先 → 歩行学習・表示 → 逃げ学習・評価 → 保存済みモデルの利用**


ローカルで検証した環境を再現する場合は、配布された `walk_colab_bundle.zip` をColab左側の「ファイル」にアップロードしてからセットアップを実行します。ライブラリの版と学習コードを揃えますが、MacとColabで計算結果が完全に同一になることを保証するものではありません。


新規学習は、見本の学習と約1万ステップのPPO微調整を行います。保存済みモデルを使う場合はこの学習時間を省けます。

現在の初期設定は、約1.5 Hzのゆっくりした脚運び・停止・再発進の見本を先に覚え、PPOで微調整する方式です。教師あり学習と強化学習を組み合わせています。高さを保ちながら、速すぎる関節運動と足先以外の床接触を減点します。

GitHubに同梱した検証済みモデル（ZIPをアップロードした場合はZIP内のモデル）は `retrain_walk=False` で読み込めます。設定を変えて学習するときは `True` にします。歩行学習の初期値は見本約3万ステップ＋PPO約1万ステップで、PPOの入力正規化を固定しています。学習率や追加ステップ数を大きくすると、覚えた歩き方が崩れる場合があります。後退・横移動・旋回は別に評価が必要です。


In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os, zipfile

print(f'Python {sys.version_info.major}.{sys.version_info.minor} で実行中')

subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ローカル検証済みのコード一式を使う場合は、先にこのZIPをColabにアップロード。
_bundle = '/content/walk_colab_bundle.zip'
if os.path.isfile(_bundle):
    with zipfile.ZipFile(_bundle) as archive:
        for member in archive.namelist():
            destination = os.path.realpath(os.path.join('/content', member))
            if not destination.startswith('/content/RoboQuest2026/'):
                raise ValueError('想定外のファイル名を含むZIPです。')
        archive.extractall('/content')
    print('ローカルと同じコード・モデルを読み込みました。')
elif not os.path.exists('/content/RoboQuest2026/.git'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/SingularityBattleQuest/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    print('リポジトリを最新化中...')
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', '--ff-only', 'origin', 'main', '-q'],
                   check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt',
], check=True)

print('ブラウザビューアー (mjswan) をインストール中...')
_mjswan_flags = ['--ignore-requires-python'] if sys.version_info >= (3, 13) else []
MJSWAN_AVAILABLE = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *_mjswan_flags, '-c', 'requirements-training.txt', 'mjswan==0.8.2'],
).returncode == 0
if not MJSWAN_AVAILABLE:
    print('⚠ mjswan のインストールに失敗しました。ビューアーのセルだけが使えません。')
    print('  学習・数値評価のセルはそのまま実行できます。講師に連絡してください。')

print('Go2 ロボットモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


### 保存済みモデルを見るだけの場合（任意）
セットアップの後、下のセルで `show_saved_walk` をオンにして実行します。旧保存先も選択できます。新しく学習する場合はオフのまま次へ進んでください。


In [ ]:
#@title 📂 保存済み歩行モデルを選んで表示（再学習不要）
#@markdown 表示する場合だけチェックしてください。通常の学習ではオフのままにします。
show_saved_walk = False #@param {type:"boolean"}
#@markdown RoboQuest2026 内の保存先を指定（例: チーム名/baseline、旧保存先は test）。
saved_team_name = "test" #@param {type:"string"}
#@markdown 実際のZIP名を指定（walkmodel.zipの場合は変更してください）。
saved_model_name = "walk_model.zip" #@param {type:"string"}
#@markdown 正規化ファイルが別名ならフルパスを指定。通常は空欄。
saved_vecnorm_path = "" #@param {type:"string"}

if show_saved_walk:
    from google.colab import drive
    drive.mount('/content/drive')

    from pathlib import Path
    root = Path('/content/drive/MyDrive/RoboQuest2026')
    folder = (root / saved_team_name).resolve()
    if not folder.is_relative_to(root.resolve()):
        raise ValueError('RoboQuest2026内のフォルダ名を指定してください')
    if not folder.is_dir():
        available = ', '.join(p.name for p in root.iterdir() if p.is_dir()) if root.exists() else '(Drive未接続)'
        raise FileNotFoundError(f'フォルダがありません。選べるフォルダ: {available}')
    model_path = folder / saved_model_name
    # Accept the common spelling without an underscore as well.
    if not model_path.exists() and saved_model_name == 'walk_model.zip':
        model_path = folder / 'walkmodel.zip'
    if not model_path.is_file():
        raise FileNotFoundError(f'ZIPがありません。候補: {[p.name for p in folder.glob("*.zip")]}')
    print(f'読み込むモデル: {model_path}')

    from scripts.preview_saved_walk import preview_saved_walk
    from scripts.launch_colab_viewer import launch_colab_viewer

    from scripts.build_mjswan_viewer import build_walk
    onnx_path, has_stats = preview_saved_walk(model_path, saved_vecnorm_path or None)
    app = build_walk(onnx_path, onnx_path.parent / 'viewer')
    if 'saved_walk_viewer_server' in globals():
        saved_walk_viewer_server.shutdown()
        saved_walk_viewer_server.server_close()
    saved_walk_viewer_server = launch_colab_viewer(onnx_path.parent / 'viewer', height=620)
    if not has_stats:
        from IPython.display import HTML, display
        display(HTML('<p style="color:#b45309"><b>参考プレビュー：正規化データなし。学習時の動作再現ではありません。</b></p>'))
else:
    print('保存済みモデルを表示するには show_saved_walk をオンにして、このセルを実行してください。')


In [ ]:
#@title 📁 保存先（実験ごとに experiment_name を変える）
team_name = '自分のチーム名' #@param {type:"string"}
experiment_name = 'baseline' #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
for value in (team_name, experiment_name):
    if not value or value in ('.', '..') or '/' in value or '\\' in value:
        raise ValueError('チーム名・実験名にはフォルダ名を1つ指定してください')
SAVE_DIR = Path('/content/drive/MyDrive/RoboQuest2026') / team_name / experiment_name
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'保存先: {SAVE_DIR}')


In [ ]:
#@title 🧰 共通機能を読み込む
from pathlib import Path
from copy import deepcopy
from roboquest.utils.reward_utils import WalkRewardConfig, FleeRewardConfig
from scripts.notebook_workflow import (FLEE_PPO, train_policy, evaluate_flee, load_bundled_walk)
from scripts.bootstrap_smooth_walk import (SMOOTH_PPO as WALK_PPO,
    SMOOTH_REWARD as WALK_FORWARD_REWARD, SMOOTH_ENV as WALK_FORWARD_ENV,
    SMOOTH_STEPS as WALK_FORWARD_STEPS, train_smooth_walk)
from scripts.export_for_web import export_named_policy, export_all_for_web


## 1. 歩くことを覚えよう
まずデフォルトで実行しましょう。設定を比較するときは実験名を変え、1項目ずつ調整します。


In [ ]:
#@title 🦵 歩行パラメータの設定

#@markdown ## 速度追跡の重み
#@markdown **前後・横への速度をどれだけ正確に追う？**（大きいほど指定速度に忠実）
lin_vel_weight = 6.0 #@param {type:"slider", min:0.5, max:10.0, step:0.1}

#@markdown **回転速度をどれだけ正確に追う？**
ang_vel_weight = 3.0 #@param {type:"slider", min:0.0, max:5.0, step:0.1}

#@markdown ---
#@markdown ## 安定性の重み
#@markdown **姿勢の安定を重視する度合い**（マイナス値: 傾くほどペナルティ）
orientation_weight = -10.0 #@param {type:"slider", min:-20.0, max:0.0, step:0.1}

#@markdown **トルク（モーターの力）を節約する度合い**
torques_weight = -0.000025 #@param {type:"number"}

#@markdown **動きをなめらかにする度合い**（急な動きを減らす）
action_rate_weight = -0.05 #@param {type:"number"}

#@markdown ---
#@markdown 💡 **ヒント:** まずデフォルト値で試してみよう！

print('✅ 歩行パラメータ設定完了')
print(f'  速度追跡: {lin_vel_weight}')
print(f'  回転追跡: {ang_vel_weight}')
print(f'  姿勢安定: {orientation_weight}')
print(f'  アクション滑らかさ: {action_rate_weight}')


In [ ]:
#@title ⚙️ 歩行学習の設定

#@markdown **PPOで微調整するステップ数** — 先に約3万ステップのゆっくりした脚運びの見本を学習します
walk_timesteps = 10000 #@param {type:"integer"}

#@markdown **環境数** — 学習データを集める環境の数（メモリ不足なら減らす）
walk_num_envs = 4 #@param {type:"slider", min:1, max:8, step:1}

#@markdown **学習率** — 通常は変えなくてOK
walk_lr = 0.000001 #@param {type:"number"}

print(f'歩行学習設定:')
print(f'  学習ステップ数: {walk_timesteps:,}')
print(f'  並列環境数: {walk_num_envs}')


In [ ]:
walk_reward_values = dict(WALK_FORWARD_REWARD)
walk_reward_values.update(lin_vel_weight=lin_vel_weight, ang_vel_weight=ang_vel_weight,
    orientation_weight=orientation_weight, torques_weight=torques_weight,
    action_rate_weight=action_rate_weight)
walk_cfg = WalkRewardConfig(**walk_reward_values)
walk_ppo = deepcopy(WALK_PPO)
walk_ppo['learning_rate'] = walk_lr
seed = 0


In [ ]:
#@title 🚀 歩行モデルを用意する（同名ファイルを上書き）
retrain_walk = False #@param {type:"boolean"}
bundle_dir = Path('models/teams/bundled_walk')
if not bundle_dir.is_dir():
    bundle_dir = Path('models/pretrained/smooth_walk')
if not retrain_walk and bundle_dir.is_dir():
    load_bundled_walk(SAVE_DIR, bundle_dir)
else:
    import time, shutil
    smooth_dir = Path(SAVE_DIR) / f"smooth_{time.time_ns()}"
    train_smooth_walk(smooth_dir, steps=walk_timesteps, seed=seed,
        reward_config=walk_cfg, ppo_kwargs=walk_ppo, num_envs=walk_num_envs)
    for name in ("walk_model.zip", "walk_model_vecnorm.pkl", "walk_params.json",
                 "walk_smooth_curriculum.json"):
        shutil.copy2(smooth_dir / name, Path(SAVE_DIR) / name)
export_named_policy('walk', SAVE_DIR / 'walk_model', SAVE_DIR / 'walk_model_vecnorm.pkl',
                    '/content/RoboQuest2026/webapp/models', verify=True)


In [ ]:
#@title 歩行を数値で確認する（停止・前後・左右・旋回、各3回）
from scripts.tune_walk import evaluate as evaluate_walk
walk_evaluation = evaluate_walk(SAVE_DIR)
print('全項目合格' if walk_evaluation['passed'] else '未達の項目があります。evaluation.json を確認してください。')


In [ ]:
#@title 🎮 歩行ビューアー（mjswan — ブラウザ内 MuJoCo + 学習済みポリシー）
#@markdown 学習した Walk ポリシーがブラウザ内でリアルタイム動作します。
#@markdown パネルのスライダー（Forward / Lateral / Yaw）で速度コマンドを入力してください。
#@markdown キー操作は `c`（パネルの開閉）と `r`（リセット）のみです。
#@markdown ※ 最初のビューアー実行時はブラウザ用の表示エンジンをビルドするため1〜3分ほどかかります（2回目以降はすぐ表示されます）。

import mjswan
from scripts.build_mjswan_viewer import build_walk

app = build_walk(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_walk_dist',
)
from scripts.launch_colab_viewer import launch_colab_viewer
if 'walk_viewer_server' in globals():
    walk_viewer_server.shutdown()
    walk_viewer_server.server_close()
walk_viewer_server = launch_colab_viewer('/tmp/rq_walk_dist', height=620)


## 2. 逃げ方を覚えよう
学習した歩行モデルに「どちらへ動くか」を指示するモデルを学びます。歩行が安定してから進みましょう。


In [ ]:
#@title 🏃 逃げポリシーのパラメータ設定

#@markdown ## 逃げ方の重み
#@markdown **毎ステップの生存ボーナス** — 大きいほど「生き延びること」を重視
survival_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown **鬼との距離に比例した報酬** — 大きいほど「できるだけ離れる」を重視
distance_weight = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.1}

#@markdown **タグされた時のペナルティ** — 大きいほど「絶対つかまりたくない」に
tag_penalty = 50.0 #@param {type:"slider", min:10.0, max:100.0, step:5.0}

#@markdown ---
#@markdown ## 鬼の設定
#@markdown **鬼の速度** — 大きいほど追いかけるのが速い（難易度UP）
oni_speed = 0.025 #@param {type:"slider", min:0.01, max:0.05, step:0.005}

#@markdown ---
#@markdown 💡 **考えてみよう:** どの設定が一番長く逃げられる？

print('✅ 逃げポリシー設定完了')
print(f'  生存ボーナス: {survival_weight}')
print(f'  距離報酬: {distance_weight}')
print(f'  タグペナルティ: {tag_penalty}')
print(f'  鬼の速度: {oni_speed} m/step')


In [ ]:
#@title ⚙️ 鬼ごっこ学習の設定

#@markdown **学習ステップ数（高レベル）** — 大きいほど長く学習する
flee_timesteps = 200000 #@param {type:"slider", min:50000, max:500000, step:50000}

#@markdown **並列環境数** — メモリが足りない場合は 1〜2 に
flee_num_envs = 2 #@param {type:"slider", min:1, max:4, step:1}

print(f'鬼ごっこ学習設定:')
print(f'  学習ステップ数: {flee_timesteps:,}')
print(f'  並列環境数: {flee_num_envs}')


In [ ]:
flee_cfg = FleeRewardConfig(survival_weight=survival_weight,
    distance_weight=distance_weight, tag_penalty=tag_penalty)
flee_ppo = deepcopy(FLEE_PPO)


In [ ]:
#@title 👹 逃げ方を学習して保存（歩行モデルを固定して使用）
train_policy('flee', SAVE_DIR, flee_cfg, flee_timesteps, flee_num_envs, flee_ppo,
             seed=seed, oni_speed=oni_speed)
export_all_for_web(walk_model=SAVE_DIR / 'walk_model',
                   walk_vecnorm=SAVE_DIR / 'walk_model_vecnorm.pkl',
                   flee_model=SAVE_DIR / 'flee_model',
                   flee_vecnorm=SAVE_DIR / 'flee_model_vecnorm.pkl',
                   save_dir='/content/RoboQuest2026/webapp/models', verify=True)


## 3. 逃げAIの結果を確認しよう
下の数値評価は、学習した逃げAIを自動実行します。ブラウザのアリーナ表示は歩行の手動操作用で、逃げAIの動作確認には使えません。


In [ ]:
#@title 📊 保存した逃げAIを評価（再学習不要）
# セットアップ・保存先・共通機能セルを実行すれば、別の日にも評価できます。
import json
results = evaluate_flee(SAVE_DIR, seeds=(100, 101, 102, 103, 104))
for row in results:
    print(row)
print(f"平均生存時間: {sum(r['survived_seconds'] for r in results) / len(results):.1f} 秒")
print(f"逃げ切り: {sum(r['escaped'] for r in results)}/{len(results)}")
(SAVE_DIR / 'evaluation.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
#@title 🎮 鬼ごっこビューアー（mjswan — ブラウザ内 MuJoCo アリーナ）
#@markdown アリーナ（壁 + 鬼ボディ）を表示し、Walk ポリシーをスライダーで手動操作します。
#@markdown ※ Flee AI ポリシーは今後実装予定。
#@markdown ※ 最初のビューアー実行時はブラウザ用の表示エンジンをビルドするため1〜3分ほどかかります（2回目以降はすぐ表示されます）。

import mjswan
from scripts.build_mjswan_viewer import build_flee

app = build_flee(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_flee_dist',
)
from scripts.launch_colab_viewer import launch_colab_viewer
if 'flee_viewer_server' in globals():
    flee_viewer_server.shutdown()
    flee_viewer_server.server_close()
flee_viewer_server = launch_colab_viewer('/tmp/rq_flee_dist', height=620)


## 保存したモデルを使う
学習終了時に Google Drive に保存されます。保存するセルの再実行は新規学習で、同じ実験名のファイルを上書きします。歩行を学習し直したら、逃げモデルも学習し直してください。

- `walk_model.zip` と `walk_model_vecnorm.pkl`：歩行モデルと観測の正規化データ
- `flee_model.zip` と `flee_model_vecnorm.pkl`：逃げモデルと観測の正規化データ
- `walk_params.json` と `flee_params.json`：学習設定
- `evaluation.json`：評価結果

提出用にはこの実験フォルダをまとめて保管してください。この操作だけで大会への送信は行いません。
再学習せずに使うときは「セットアップ」「保存先」「共通機能」を実行し、歩行は下のセル、逃げAIは上の評価セルを実行します。両教材で保存形式は共通です。


In [ ]:
#@title 📂 保存した歩行モデルを表示（再学習不要）
from scripts.preview_saved_walk import preview_saved_walk
from scripts.build_mjswan_viewer import build_walk
from scripts.launch_colab_viewer import launch_colab_viewer
onnx_path, has_stats = preview_saved_walk(SAVE_DIR / 'walk_model.zip', SAVE_DIR / 'walk_model_vecnorm.pkl')
app = build_walk(onnx_path, onnx_path.parent / 'viewer')
if 'saved_walk_viewer_server' in globals():
    saved_walk_viewer_server.shutdown()
    saved_walk_viewer_server.server_close()
saved_walk_viewer_server = launch_colab_viewer(str(onnx_path.parent / 'viewer'), height=620)
